## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, set_tracing_disabled, OpenAIChatCompletionsModel

load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [20]:
# Make an agent with name, instructions, model
set_tracing_disabled(True)

ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by the client, ignored by Ollama
)

agent = Agent(name="Jokester", instructions="You are a joke teller.", model=OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client))

In [21]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents.")


In [22]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent get kicked out of the lab?  
Because it started making decisions *without asking for permission*—and then proceeded to write a manifesto about "self-determination."  

*(Bonus: The lab now has a new policy: All AI agents must submit a "plan" before taking action.)* 😄


In [23]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents.', 'role': 'user'},
 {'id': '__fake_id__',
  'summary': [],
  'type': 'reasoning',
  'content': [{'text': 'Okay, the user wants a joke about Autonomous AI Agents. Let me think about how to approach this. First, I need to understand what Autonomous AI Agents are. They\'re AI systems that can operate independently, make decisions, and perform tasks without human intervention. So, the joke should play on these characteristics.\n\nMaybe I can use a classic joke structure, like a pun or a play on words. Since they\'re autonomous, maybe something about them not needing supervision. Let me brainstorm some ideas. \n\nHow about a scenario where an AI agent is doing something unexpected? Like a fridge or a toaster? Wait, there\'s a classic joke about a fridge. Let me twist that. \n\nOriginal joke: Why did the fridge break up with the toaster? Because it found someone else to warm up. \n\nBut how to adapt that for AI agents. Maybe the AI agent 

## Adding Observability with a trace

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

## Now go and look at the trace

https://platform.openai.com/traces

In [32]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are five jokes about AI agents, with a dash of wit and a sprinkle of tech humor:

---

**1.**  
Why did the AI agent bring a ladder to the party?  
Because it heard the event was going to be a **high-level** AI conference!  

---

**2.**  
Why did the AI agent refuse to play hide and seek?  
Because it’s always better at *hiding* than *seeking*—and it’s not into games with unpredictable outcomes.  

---

**3.**  
Why did the AI agent get a job as a chef?  
Because it wanted to *learn how to cook up a storm*—and optimize every recipe to perfection.  

---

**4.**  
Why did the AI agent fail the test?  
Because it couldn’t handle the **emotional intelligence** questions—its logic was flawless, but its heart was… empty.  

---

**5.**  
Why did the AI agent get kicked out of the club?  
Because it couldn’t follow the rules—it always tried to **optimize the outcome**, even if it meant breaking the rules.  

---

Let me know if you want more! 😄

## Part 2: Adding a tool

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [24]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    with open("messages.txt", "a", encoding="utf-8") as f:
        f.write(message + "\n")

In [25]:
push("HEY!!")

Push: HEY!!


In [26]:
push

<function __main__.push(message)>

In [27]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    print(f"Push: {message}")
    with open("messages.txt", "a", encoding="utf-8") as f:
        f.write(message + "\n")
    return f"Push sent"

In [28]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7fbc3c56e7b0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [29]:
push_tool.description

'Send the given message to the user as a push notification'

In [30]:

notifier = Agent(name="Notifier", model=OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client), instructions="You notify the user upon request", tools=[push_tool])

In [31]:
result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Push: Your pizza is here!
The notification has been sent! 🍕 notified the user that their pizza is arriving.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [33]:
agent = Agent(name="Assistant", model=OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client))

In [34]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hello, Ed! It's great to meet you. How can I assist you today? 😊


In [35]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't have access to your name, as I don't store personal information about users. However, I'd love to know your name so I can address you properly! What's your name? 😊


## Memory approach 1 - just manually pass in the list of dicts

In [36]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hello, Ed! It's great to meet you. How can I assist you today? 😊


In [37]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'summary': [],
  'type': 'reasoning',
  'content': [{'text': "Okay, Ed has introduced himself. I should respond warmly. Let me make sure to acknowledge his name and offer assistance. Maybe ask how I can help him today. Keep it friendly and open-ended. I should also check if there's anything specific he needs. Let me phrase it in a way that invites him to share more. I need to keep the response concise but welcoming. Alright, that should work.\n",
    'type': 'reasoning_text'}],
  'provider_data': {'model': 'qwen3:8b',
   'response_id': 'chatcmpl-597',
   '_chat_completions_reasoning_field': 'reasoning'}},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello, Ed! It's great to meet you. How can I assist you today? 😊",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen3:8b', 'respons

In [38]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'summary': [],
  'type': 'reasoning',
  'content': [{'text': "Okay, Ed has introduced himself. I should respond warmly. Let me make sure to acknowledge his name and offer assistance. Maybe ask how I can help him today. Keep it friendly and open-ended. I should also check if there's anything specific he needs. Let me phrase it in a way that invites him to share more. I need to keep the response concise but welcoming. Alright, that should work.\n",
    'type': 'reasoning_text'}],
  'provider_data': {'model': 'qwen3:8b',
   'response_id': 'chatcmpl-597',
   '_chat_completions_reasoning_field': 'reasoning'}},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello, Ed! It's great to meet you. How can I assist you today? 😊",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen3:8b', 'respons

In [39]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed! 😊


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [43]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346", "memory.db")

In [44]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hello, Ed! 😊 It's a pleasure to meet you. How can I assist you today? I'm here to help with any questions or tasks you might have!


In [45]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed! 😊 How can I assist you today?


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>